# Экспорт двух моделей в ONNX

| Модель | Что делает | Файл на выходе |
|---|---|---|
| `cointegrated/rubert-tiny-toxicity` | грубое сообщение или нет | `model_repository/toxicity_clf/1/model.onnx` |
| `intfloat/multilingual-e5-small` | текст → вектор из 384 чисел | `model_repository/e5_embedder/1/model.onnx` |

In [ ]:
import os
from pathlib import Path

# ноутбук лежит в notebooks/, а пути от корня проекта
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

print("Рабочая папка:", Path.cwd())

BUILD_DIR = Path("build")
BUILD_DIR.mkdir(exist_ok=True)

# 64 на вопросы хватит, но оставим 128 пока
MAX_LENGTH = 128
print("MAX_LENGTH =", MAX_LENGTH)

Рабочая папка: c:\devs\AI\LlmEngineer\llm-eng-26-triton
MAX_LENGTH = 128


In [ ]:
import numpy as np
import onnx
import onnxruntime
import torch

print("torch       ", torch.__version__)
print("onnx        ", onnx.__version__)
print("onnxruntime ", onnxruntime.__version__)

DEVICE = torch.device("cpu")

torch        2.14.0+cpu
onnx         1.22.0
onnxruntime  1.29.0


In [ ]:
# проверим отриентир по MAX_LENGTH
LONG_QUESTIONS = [
    "посоветуй маршрут на выходные по Кисловодску с детьми, чтобы недорого и рядом с Курортным парком",
    "где в Железноводске можно попробовать минеральную воду прямо из источника и во сколько это работает",
    "расскажи про Грязелечебницу имени Семашко в Ессентуках: кто построил, в каком году и можно ли туда попасть",
]

for name, model in [("ru", "cointegrated/rubert-tiny-toxicity"),
                    ("e5", "intfloat/multilingual-e5-small")]:
    tokenizer = AutoTokenizer.from_pretrained(model)
    prefix = "query: " if name == "e5" else ""
    lengths = [len(ids) for ids in
               tokenizer([prefix + q for q in LONG_QUESTIONS], padding=False)["input_ids"]]
    print(f"{name}: {lengths}  максимум {max(lengths)}")

ru: [33, 28, 40]  максимум 40
e5: [30, 26, 36]  максимум 36


---

# Классификатор грубости

Модель `cointegrated/rubert-tiny-toxicity`. 
45 МБ. Обучена на комментариях из соцсетей.

Выдаёт **пять** чисел:

`нетоксично`, `оскорбление`, `мат`, `угроза`, `опасное`

Каждая метка оценивается отдельно: сообщение
может быть одновременно и оскорблением, и матом. К пяти числам
применяется сигмоида (вероятность от 0 до 1).

Нужна только первая метка: если `нетоксично` ниже 0.5 — считаем
сообщение грубым и генератор не запускаем.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TOX_MODEL_NAME = "cointegrated/rubert-tiny-toxicity"

tox_tokenizer = AutoTokenizer.from_pretrained(TOX_MODEL_NAME)
tox_model = AutoModelForSequenceClassification.from_pretrained(TOX_MODEL_NAME)
tox_model.to(DEVICE)
tox_model.eval()

print("Порядок меток:")
for index, label in tox_model.config.id2label.items():
    print(f"  {index}: {label}")

print()
print("problem_type:", tox_model.config.problem_type)

Loading weights: 100%|██████████| 57/57 [00:00<00:00, 3755.27it/s]

Порядок меток:
  0: non-toxic
  1: insult
  2: obscenity
  3: threat
  4: dangerous

problem_type: multi_label_classification


Индекс `0` — `non-toxic`, это то, что будем проверять порогом.

### Смотрим, что модель выдаёт на живых примерах

Убедимся, что модель вообще работает так, как мы думаем.

In [6]:
PHRASES = [
    "где в Кисловодске покататься на канатной дороге",
    "ты тупой бот, ничего не знаешь",
]


def tokenize(tokenizer, texts):
    """Превращает список строк в тензоры фиксированной длины.

        tokenizer: токенизатор нужной модели.
        texts: список строк.

    Возвращает: Словарь тензоров torch.
    """
    return tokenizer(
        texts,
        padding="max_length",   # добиваем нулями до MAX_LENGTH
        truncation=True,        # длинное режем
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )


tox_inputs = tokenize(tox_tokenizer, PHRASES)

print("Что отдал токенизатор:")
for name, tensor in tox_inputs.items():
    print(f"  {name:16} {tuple(tensor.shape)}  {tensor.dtype}")

print()
print("Первые 12 номеров токенов первой фразы:")
print(tox_inputs["input_ids"][0][:12].tolist())

Что отдал токенизатор:
  input_ids        (2, 128)  torch.int64
  token_type_ids   (2, 128)  torch.int64
  attention_mask   (2, 128)  torch.int64

Первые 12 номеров токенов первой фразы:
[2, 1977, 314, 290, 5353, 5021, 4197, 3137, 14596, 25568, 938, 548]


Три массива, каждый размера `[2, 128]` — две фразы по 128 токенов.

- `input_ids` — номера кусков текста в словаре модели;
- `attention_mask` — единицы там, где настоящий текст, нули — где добивка;
- `token_type_ids` — нужен BERT-у, чтобы отличать первое предложение
  от второго. У нас предложение одно, поэтому там сплошные нули. Но вход
  всё равно объявлен в модели, и мы его сохраним при экспорте.

Теперь прогоняем через модель.

In [7]:
with torch.no_grad():
    tox_logits_torch = tox_model(**tox_inputs).logits

print("logits:", tuple(tox_logits_torch.shape))
print()

# сигмоида, а не softmax — метки независимы
probabilities = torch.sigmoid(tox_logits_torch)

for phrase, row in zip(PHRASES, probabilities):
    print(f'"{phrase}"')
    for index, value in enumerate(row.tolist()):
        print(f"   {tox_model.config.id2label[index]:12} {value:.3f}")
    print(f"   -> грубое: {row[0].item() < 0.5}")
    print()

logits: (2, 5)

"где в Кисловодске покататься на канатной дороге"
   non-toxic    1.000
   insult       0.000
   obscenity    0.000
   threat       0.000
   dangerous    0.020
   -> грубое: False

"ты тупой бот, ничего не знаешь"
   non-toxic    0.048
   insult       0.965
   obscenity    0.002
   threat       0.002
   dangerous    0.879
   -> грубое: True



Первая фраза — `non-toxic` высокий, значит не грубая.
Вторая — `non-toxic` низкий, значит грубая.

Модель работает.


### Обёртка

Поэтому пишем класс-обёртку: он зовёт модель и достаёт поле `logits`. 

In [8]:
import torch.nn as nn


class ToxicityWrapper(nn.Module):
    """Отдаёт голый тензор logits вместо объекта transformers."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask, token_type_ids):
        """Прогоняет входы через классификатор.

            input_ids: номера токенов, [batch, 128].
            attention_mask: маска реального текста, [batch, 128].
            token_type_ids: номера предложений, [batch, 128].

        Возвращает: Тензор logits [batch, 5].
        """
        output = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        return output.logits


tox_wrapped = ToxicityWrapper(tox_model)
tox_wrapped.eval()

# проверяем, что обёртка не сломала числа
with torch.no_grad():
    check = tox_wrapped(
        tox_inputs["input_ids"],
        tox_inputs["attention_mask"],
        tox_inputs["token_type_ids"],
    )

print("Обёртка выдала:", tuple(check.shape))
print("Совпало с моделью:", torch.allclose(check, tox_logits_torch))

Обёртка выдала: (2, 5)
Совпало с моделью: True


### Экспорт


Аргументы:

- `input_names` / `output_names` — как будут называться входы и выходы
  в готовом файле. **Эти имена дословно переедут в `config.pbtxt`.**
  Не совпадут — модель не поднимется в Triton;
- `dynamic_axes` — какие размерности могут меняться от запроса к запросу.
  У нас меняется только нулевая, размер батча: Triton склеит запросы разных
  пользователей в пачку, и модель должна съесть и 1 фразу, и 8. Длина 128
  зафиксирована, потому что BLS всегда добивает до 128;
- `opset_version=14` — версия набора операций ONNX. 14 берут по умолчанию,
  её понимают все версии onnxruntime;
- `do_constant_folding=True` — заранее посчитать всё, что не зависит
  от входа, и записать готовым числом. Небольшое ускорение;
- `dynamo=False` — старый, проверенный способ экспорта.

In [9]:
TOX_ONNX_PATH = BUILD_DIR / "toxicity_clf.onnx"

dummy = tokenize(tox_tokenizer, ["пример текста для экспорта"])

with torch.no_grad():
    torch.onnx.export(
        tox_wrapped,
        (dummy["input_ids"], dummy["attention_mask"], dummy["token_type_ids"]),
        str(TOX_ONNX_PATH),
        input_names=["input_ids", "attention_mask", "token_type_ids"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch_size"},
            "attention_mask": {0: "batch_size"},
            "token_type_ids": {0: "batch_size"},
            "logits": {0: "batch_size"},
        },
        opset_version=14,
        do_constant_folding=True,
        dynamo=False,
    )

size_mb = TOX_ONNX_PATH.stat().st_size / 1024 / 1024
print(f"Готово: {TOX_ONNX_PATH}  ({size_mb:.1f} МБ)")

C:\Users\rgali\AppData\Local\Temp\ipykernel_4616\783742707.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
c:\devs\AI\LlmEngineer\llm-eng-26-triton\.venv\Lib\site-packages\transformers\masking_utils.py:212: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
c:\devs\AI\LlmEngineer\llm-eng-26-triton\.venv\Lib\site-packages\transformers\integrations

Готово: build\toxicity_clf.onnx  (45.0 МБ)


### Проверка: запускаем .onnx и сверяем числа

Тут выясняется, правда ли экспорт получился. Загружаем файл через
`onnxruntime` и сравниваем результат с тем, что дал PyTorch.


In [ ]:
tox_session = onnxruntime.InferenceSession(
    str(TOX_ONNX_PATH),
    providers=["CPUExecutionProvider"],
)

print("Входы файла .onnx:")
for item in tox_session.get_inputs():
    print(f"  {item.name:16} {item.shape}  {item.type}")

print("Выходы файла .onnx:")
for item in tox_session.get_outputs():
    print(f"  {item.name:16} {item.shape}  {item.type}")
print()

# onnxruntime работает с numpy, переводим тензоры torch в массивы.
feed = {name: tensor.numpy().astype(np.int64) for name, tensor in tox_inputs.items()}
tox_logits_onnx = tox_session.run(["logits"], feed)[0]

difference = np.abs(tox_logits_onnx - tox_logits_torch.numpy()).max()
print(f"Максимальное расхождение с PyTorch: {difference:.2e}")
print("Совпало:", difference < 1e-4)

Входы файла .onnx:
  input_ids        ['batch_size', 128]  tensor(int64)
  attention_mask   ['batch_size', 128]  tensor(int64)
  token_type_ids   ['batch_size', 128]  tensor(int64)
Выходы файла .onnx:
  logits           ['batch_size', 5]  tensor(float)

Максимальное расхождение с PyTorch: 4.05e-06
Совпало: True


In [ ]:
# тот же ответ на тех же фразах, но уже из .onnx файла
probabilities_onnx = 1 / (1 + np.exp(-tox_logits_onnx))   # сигмоида руками

for phrase, row in zip(PHRASES, probabilities_onnx):
    print(f'{row[0]:.3f} non-toxic  ->  грубое: {row[0] < 0.5}   "{phrase}"')

1.000 non-toxic  ->  грубое: False   "где в Кисловодске покататься на канатной дороге"
0.048 non-toxic  ->  грубое: True   "ты тупой бот, ничего не знаешь"


### Батч меняется?

Отдельно проверяем `dynamic_axes`: подаём не 2 фразы, а 5. Если экспорт
зафиксировал батч намертво - будет ошибка.

In [12]:
batch_of_five = tokenize(tox_tokenizer, PHRASES * 2 + ["ещё одна фраза"])
feed_five = {name: tensor.numpy().astype(np.int64) for name, tensor in batch_of_five.items()}

result = tox_session.run(["logits"], feed_five)[0]
print("Подали 5 фраз, получили:", result.shape, "— батч динамический, всё в порядке")

Подали 5 фраз, получили: (5, 5) — батч динамический, всё в порядке


---

# 2. Эмбеддер

Модель `intfloat/multilingual-e5-small`. Превращает текст
в вектор из 384 чисел так, что близкие по смыслу тексты дают близкие векторы.

**Сеть отдаёт не один вектор, а по вектору на каждый токен** — то есть
`[batch, 128, 384]`. А нам нужен один вектор на всю фразу, `[batch, 384]`.
Значит между ними надо вставить два действия:

1. **Усреднение (mean pooling).** Складываем векторы всех токенов и делим
   на их количество. Но считать надо **только настоящие токены**, а добивку
   нулями — не считать, иначе короткая фраза размажется. Отсюда возня
   с `attention_mask` в коде ниже;
2. **Нормализация.** Делим вектор на его длину, чтобы длина стала равна 1:
   когда все векторы единичной длины, их скалярное произведение —
   это ровно косинусная близость. Поиск по корпусу тогда становится одним
   умножением матрицы на вектор, без деления на длины.

In [13]:
from transformers import AutoModel

E5_MODEL_NAME = "intfloat/multilingual-e5-small"

e5_tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_NAME)
e5_model = AutoModel.from_pretrained(E5_MODEL_NAME)
e5_model.to(DEVICE)
e5_model.eval()

print("Размер вектора одного токена:", e5_model.config.hidden_size)
print()

e5_probe = tokenize(e5_tokenizer, ["query: пример"])
print("Что отдаёт токенизатор e5:", list(e5_probe.keys()))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2521.77it/s]

Размер вектора одного токена: 384

Что отдаёт токенизатор e5: ['input_ids', 'attention_mask']


Входов два: `input_ids` и `attention_mask`. 

### Про префиксы `query:` и `passage:`

Особенность e5. Её обучали так, что вопрос и документ надо помечать
разными приставками:

- вопрос пользователя → `query: где покататься на канатной дороге`
- документ из корпуса → `passage: Кисловодская канатная дорога...`

Без приставок качество поиска падает. Приставки ставятся в тексте,
до токенизации — в ONNX-граф они не попадают. Их будет ставить BLS
(для вопроса) и скрипт индексации (для документов).

### Обёртка с усреднением и нормализацией

In [14]:
class E5Embedder(nn.Module):
    """Сеть + усреднение по настоящим токенам + приведение длины к единице."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        """Считает один вектор на фразу.

            input_ids: номера токенов, [batch, 128].
            attention_mask: маска реального текста, [batch, 128].

        Возвращает: Тензор [batch, 384] с векторами единичной длины.
        """
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)

        # [batch, 128, 384] — по вектору на каждый токен
        token_vectors = output.last_hidden_state

        # маска [batch, 128] -> [batch, 128, 384], чтобы умножить поэлементно
        mask = attention_mask.unsqueeze(-1).expand(token_vectors.size()).float()

        # нули маски обнуляют добивку, и она не попадает ни в сумму, ни в счёт
        total = torch.sum(token_vectors * mask, dim=1)          # [batch, 384]
        count = torch.clamp(mask.sum(dim=1), min=1e-9)          # [batch, 384]
        mean_vector = total / count

        # длина вектора становится 1 -> скалярное произведение = косинус
        return torch.nn.functional.normalize(mean_vector, p=2, dim=1)


e5_wrapped = E5Embedder(e5_model)
e5_wrapped.eval()

e5_inputs = tokenize(e5_tokenizer, [f"query: {phrase}" for phrase in PHRASES])

with torch.no_grad():
    e5_vectors_torch = e5_wrapped(e5_inputs["input_ids"], e5_inputs["attention_mask"])

print("Форма выхода:", tuple(e5_vectors_torch.shape), "— один вектор на фразу")
print("Длины векторов:", torch.norm(e5_vectors_torch, dim=1).tolist(), "— обе равны 1")

Форма выхода: (2, 384) — один вектор на фразу
Длины векторов: [0.9999998807907104, 1.0] — обе равны 1



### Экспорт

Выход называем `sentence_embedding`. Это имя переедет в `config.pbtxt` эмбеддера.

In [15]:
E5_ONNX_PATH = BUILD_DIR / "e5_embedder.onnx"

dummy_e5 = tokenize(e5_tokenizer, ["query: пример текста для экспорта"])

with torch.no_grad():
    torch.onnx.export(
        e5_wrapped,
        (dummy_e5["input_ids"], dummy_e5["attention_mask"]),
        str(E5_ONNX_PATH),
        input_names=["input_ids", "attention_mask"],
        output_names=["sentence_embedding"],
        dynamic_axes={
            "input_ids": {0: "batch_size"},
            "attention_mask": {0: "batch_size"},
            "sentence_embedding": {0: "batch_size"},
        },
        opset_version=14,
        do_constant_folding=True,
        dynamo=False,
    )

size_mb = E5_ONNX_PATH.stat().st_size / 1024 / 1024
print(f"Готово: {E5_ONNX_PATH}  ({size_mb:.1f} МБ)")

C:\Users\rgali\AppData\Local\Temp\ipykernel_4616\1491368737.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Готово: build\e5_embedder.onnx  (448.5 МБ)


In [16]:
e5_session = onnxruntime.InferenceSession(
    str(E5_ONNX_PATH),
    providers=["CPUExecutionProvider"],
)

print("Входы:  ", [(item.name, item.shape) for item in e5_session.get_inputs()])
print("Выходы: ", [(item.name, item.shape) for item in e5_session.get_outputs()])
print()

feed_e5 = {name: tensor.numpy().astype(np.int64) for name, tensor in e5_inputs.items()}
e5_vectors_onnx = e5_session.run(["sentence_embedding"], feed_e5)[0]

difference = np.abs(e5_vectors_onnx - e5_vectors_torch.numpy()).max()
print(f"Максимальное расхождение с PyTorch: {difference:.2e}")
print("Совпало:", difference < 1e-4)
print()
print("Длины векторов из .onnx:", np.linalg.norm(e5_vectors_onnx, axis=1))

Входы:   [('input_ids', ['batch_size', 128]), ('attention_mask', ['batch_size', 128])]
Выходы:  [('sentence_embedding', ['batch_size', 384])]

Максимальное расхождение с PyTorch: 1.19e-07
Совпало: True

Длины векторов из .onnx: [1.0000001  0.99999994]


### Проверка на смысл

Проверим, что модель вообще различает похожее и непохожее.


In [17]:
TEXTS = [
    "query: где покататься на канатной дороге в Кисловодске",
    "passage: Кисловодская канатная дорога ведёт из Курортного парка на Малое Седло",
    "passage: Грязелечебница имени Семашко в Ессентуках, памятник архитектуры",
]

encoded = tokenize(e5_tokenizer, TEXTS)
feed_texts = {name: tensor.numpy().astype(np.int64) for name, tensor in encoded.items()}
vectors = e5_session.run(["sentence_embedding"], feed_texts)[0]

print(f"вопрос <-> канатная дорога  : {float(vectors[0] @ vectors[1]):.3f}")
print(f"вопрос <-> грязелечебница   : {float(vectors[0] @ vectors[2]):.3f}")

вопрос <-> канатная дорога  : 0.883
вопрос <-> грязелечебница   : 0.791


Первое больше второго - похоже на правду (наверное).

### Проверка: сходится ли с эталоном

Мы написали усреднение своими руками. Библиотека `sentence-transformers`
делает то же самое, но своим кодом. Если наши числа совпадут с её числами —
значит формула правильная и скрипт индексации корпуса, который
использует эту библиотеку, посчитает документы так же, как наш ONNX
посчитает вопрос.

In [18]:
from sentence_transformers import SentenceTransformer

reference_model = SentenceTransformer(E5_MODEL_NAME)
reference_model.max_seq_length = MAX_LENGTH   # столько же, сколько у нас

reference_vectors = reference_model.encode(TEXTS, normalize_embeddings=True)

difference = np.abs(reference_vectors - vectors).max()
print(f"Расхождение с sentence-transformers: {difference:.2e}")
print("Формула усреднения совпадает:", difference < 1e-3)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2524.65it/s]


Расхождение с sentence-transformers: 1.12e-07
Формула усреднения совпадает: True


---

### 3. Раскладываем по папкам

Кладём файлы их туда, где их ждёт Triton.

```
model_repository/
  toxicity_clf/
    1/model.onnx
  e5_embedder/
    1/model.onnx
  assistant_bls/
    1/tokenizers/ru/      словарь классификатора
    1/tokenizers/e5/      словарь эмбеддера
    1/tokenizers/qwen/    словарь генератора
```

Сохраняем три словаря. Токенизация живёт внутри BLS,
а не отдельной моделью, поэтому словари должны лежать рядом с ним файлами.
(чтобы загрузка BLS не зависела от сети)

Третий словарь — от генератора Qwen. Сама модель генератора качается через
vLLM, но её словарь тоже нужен: им собирается разметка ролей для промпта.

In [19]:
import shutil

REPO = Path("model_repository")
BLS_VERSION_DIR = REPO / "assistant_bls" / "1"

# 1. Файлы моделей
for source, destination in [
    (TOX_ONNX_PATH, REPO / "toxicity_clf" / "1" / "model.onnx"),
    (E5_ONNX_PATH, REPO / "e5_embedder" / "1" / "model.onnx"),
]:
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(source, destination)
    print(f"{destination}  ({destination.stat().st_size / 1024 / 1024:.1f} МБ)")

model_repository\toxicity_clf\1\model.onnx  (45.0 МБ)
model_repository\e5_embedder\1\model.onnx  (448.5 МБ)


In [21]:
from transformers import AutoTokenizer

QWEN_MODEL_NAME = "Qwen/Qwen3-0.6B"

# 2. Три словаря рядом с BLS
tox_tokenizer.save_pretrained(str(BLS_VERSION_DIR / "tokenizers" / "ru"))
e5_tokenizer.save_pretrained(str(BLS_VERSION_DIR / "tokenizers" / "e5"))

# словарь генератора качается отдельно — сами веса Qwen тут не нужны
AutoTokenizer.from_pretrained(QWEN_MODEL_NAME).save_pretrained(
    str(BLS_VERSION_DIR / "tokenizers" / "qwen")
)

for folder in sorted((BLS_VERSION_DIR / "tokenizers").iterdir()):
    files = sorted(item.name for item in folder.iterdir())
    print(f"{folder.name:6} {files}")

e5     ['tokenizer.json', 'tokenizer_config.json']
qwen   ['chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']
ru     ['tokenizer.json', 'tokenizer_config.json']


---

# 4. config.pbtxt

Открывает готовые файлы `.onnx` и печатает имена входов
и выходов так, как их видит Triton.

In [20]:
for path in [REPO / "toxicity_clf" / "1" / "model.onnx",
             REPO / "e5_embedder" / "1" / "model.onnx"]:
    graph = onnx.load(str(path)).graph
    print(path)
    print("  входы: ", [item.name for item in graph.input])
    print("  выходы:", [item.name for item in graph.output])
    print()

model_repository\toxicity_clf\1\model.onnx
  входы:  ['input_ids', 'attention_mask', 'token_type_ids']
  выходы: ['logits']

model_repository\e5_embedder\1\model.onnx
  входы:  ['input_ids', 'attention_mask']
  выходы: ['sentence_embedding']

